# Forecasting with Litterman

This tutorial shows how to produce probabilistic forecasts from a fitted Bayesian VAR.

In [1]:
import numpy as np
import pandas as pd

from litterman import VAR, VARData
from litterman.samplers import NUTSSampler

## Simulate and fit

We'll create a simple VAR(1) and fit it.

In [2]:
rng = np.random.default_rng(42)
T = 200
y = np.zeros((T, 2))
for t in range(1, T):
    y[t] = 0.5 * y[t - 1] + rng.standard_normal(2) * 0.1

index = pd.date_range("2000-01-01", periods=T, freq="QS")
data = VARData(endog=y, endog_names=["gdp", "inflation"], index=index)

sampler = NUTSSampler(draws=500, tune=500, chains=2, cores=1, random_seed=42)
fitted = VAR(lags=1, prior="minnesota").fit(data, sampler=sampler)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, B, sigma_sd, tril_offdiag]


/Users/thomaspinder/Developer/litterman/.venv/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 2 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


## Produce forecasts

Call `.forecast(steps=h)` to get an h-step-ahead forecast.

In [3]:
fcast = fitted.forecast(steps=8)
fcast.median()

,gdp,inflation
0,0.044380,0.017550
1,0.028588,0.008186
2,0.018957,0.002538
3,0.012780,-0.000982
4,0.009412,-0.003149
5,0.007440,-0.004240
6,0.006211,-0.004885
7,0.005311,-0.005509


## Credible intervals

Use `.hdi()` to get highest density intervals.

In [4]:
hdi = fcast.hdi(prob=0.89)
print("Lower bounds:")
print(hdi.lower)
print("\nUpper bounds:")
print(hdi.upper)

Lower bounds:
        gdp  inflation
0  0.030587   0.005467
1  0.008050  -0.007335
2 -0.004244  -0.017074
3 -0.011505  -0.021446
4 -0.014958  -0.024751
5 -0.017830  -0.025826
6 -0.020278  -0.028067
7 -0.021695  -0.028037

Upper bounds:
        gdp  inflation
0  0.056032   0.029566
1  0.047139   0.029426
2  0.042489   0.026682
3  0.039606   0.025712
4  0.039571   0.024462
5  0.038862   0.024505
6  0.037722   0.022896
7  0.037184   0.023374


## Convert to DataFrame

Use `.to_dataframe()` for a tidy format suitable for further analysis.

In [5]:
fcast.to_dataframe()

,gdp,inflation
step,,
0,0.044380,0.017550
1,0.028588,0.008186
2,0.018957,0.002538
3,0.012780,-0.000982
4,0.009412,-0.003149
5,0.007440,-0.004240
6,0.006211,-0.004885
7,0.005311,-0.005509
